# Task 02 — Bonus Challenge: Scikit-learn Compatible Cleaning Transformer

**Student:** Mussa Khan ([@ahaseeb003](https://github.com/ahaseeb003))
**Program:** 3-Month Machine Learning Internship — Month 1

**Challenge:** Wrap the cleaning pipeline from `Task_02_Advanced_Data_Cleaning_and_Preprocessing.ipynb`
in a scikit-learn `BaseEstimator` + `TransformerMixin` class so it plugs directly into a
scikit-learn `Pipeline`.

This notebook is self-contained — it regenerates the same messy synthetic dataset,
then shows the cleaning logic as a proper `fit` / `transform` estimator.


## 1. Setup & Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/Task_02_Advanced_Data_Cleaning_and_Preprocessing'
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder ready at:", PROJECT_DIR)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
np.random.seed(42)


## 2. Recreate (or Load) the Messy Dataset

If `raw_customer_data.csv` already exists in your Drive project folder (from the main
notebook), we load it. Otherwise we regenerate it here so this notebook also runs standalone.


In [ ]:
raw_path = os.path.join(PROJECT_DIR, "raw_customer_data.csv")

if os.path.exists(raw_path):
    raw_df = pd.read_csv(raw_path)
    print("Loaded existing raw data from Drive:", raw_path)
else:
    n = 1000
    raw_df = pd.DataFrame({
        "customer_id": range(1, n + 1),
        "age": np.random.randint(18, 70, size=n).astype(float),
        "income": np.random.normal(loc=55000, scale=15000, size=n).round(2),
        "purchase_amount": np.random.exponential(scale=120, size=n).round(2),
        "signup_date": pd.date_range("2021-01-01", periods=n, freq="D").astype(str),
        "category": np.random.choice(["Electronics", "Clothing", "Groceries", "Books"], size=n),
        "rating": np.random.randint(1, 6, size=n).astype(float),
    })

    for col in ["age", "income", "purchase_amount", "rating", "category"]:
        missing_idx = np.random.choice(raw_df.index, size=int(n * 0.07), replace=False)
        raw_df.loc[missing_idx, col] = np.nan

    outlier_idx = np.random.choice(raw_df.index, size=15, replace=False)
    raw_df.loc[outlier_idx, "income"] = raw_df.loc[outlier_idx, "income"] * 8
    raw_df.loc[outlier_idx[:5], "age"] = np.random.choice([150, 200, -5], size=5)

    mixed_type_idx = np.random.choice(raw_df.index, size=20, replace=False)
    raw_df.loc[mixed_type_idx, "age"] = raw_df.loc[mixed_type_idx, "age"].astype(str) + "yrs"

    duplicates = raw_df.sample(25, random_state=1)
    raw_df = pd.concat([raw_df, duplicates], ignore_index=True)

    raw_df.to_csv(raw_path, index=False)
    print("No existing file found — generated a fresh raw dataset and saved it to Drive.")

print("Shape:", raw_df.shape)
raw_df.head()


## 3. The Custom Transformer

`DataCleaningTransformer` implements the same five steps as the main notebook
(fix types → validate schema → drop duplicates → cap outliers → impute missing values),
but packaged as a proper scikit-learn estimator:

- `fit(X, y=None)` learns anything the transform step needs from the **training** data only
  (here: the median/mode values used for imputation, and the IQR bounds for outlier capping).
  This avoids **data leakage** from test data into training statistics.
- `transform(X)` applies those learned values to any new data (train, validation, or test).


In [ ]:
class DataCleaningTransformer(BaseEstimator, TransformerMixin):
    """Scikit-learn compatible transformer that cleans the customer records dataset.

    Parameters
    ----------
    numeric_cols : list of str
        Numeric columns to impute (median) and cap outliers on.
    categorical_cols : list of str
        Categorical columns to impute using the mode.
    iqr_factor : float, default=1.5
        Multiplier for the IQR outlier-capping rule.
    """

    def __init__(self, numeric_cols=None, categorical_cols=None, iqr_factor=1.5):
        self.numeric_cols = numeric_cols or ["age", "income", "purchase_amount", "rating"]
        self.categorical_cols = categorical_cols or ["category"]
        self.iqr_factor = iqr_factor

    def _fix_types(self, X):
        X = X.copy()
        if "age" in X.columns:
            X["age"] = (
                X["age"].astype(str).str.extract(r"(-?\d+\.?\d*)")[0].astype(float)
            )
        if "signup_date" in X.columns:
            X["signup_date"] = pd.to_datetime(X["signup_date"], errors="coerce")
        for col in self.categorical_cols:
            if col in X.columns:
                X[col] = X[col].astype("category")
        return X

    def _validate_schema(self, X):
        X = X.copy()
        if "age" in X.columns:
            X.loc[(X["age"] < 0) | (X["age"] > 110), "age"] = np.nan
        if "income" in X.columns:
            X.loc[X["income"] < 0, "income"] = np.nan
        if "purchase_amount" in X.columns:
            X.loc[X["purchase_amount"] < 0, "purchase_amount"] = np.nan
        if "rating" in X.columns:
            X.loc[(X["rating"] < 1) | (X["rating"] > 5), "rating"] = np.nan
        return X

    def fit(self, X, y=None):
        # Learn imputation + outlier statistics from the training data ONLY
        X = self._fix_types(X)
        X = self._validate_schema(X)

        self.medians_ = {}
        self.bounds_ = {}
        for col in self.numeric_cols:
            if col not in X.columns:
                continue
            self.medians_[col] = X[col].median()

            q1, q3 = X[col].quantile(0.25), X[col].quantile(0.75)
            iqr = q3 - q1
            self.bounds_[col] = (q1 - self.iqr_factor * iqr, q3 + self.iqr_factor * iqr)

        self.modes_ = {}
        for col in self.categorical_cols:
            if col not in X.columns:
                continue
            mode_val = X[col].mode(dropna=True)
            self.modes_[col] = mode_val[0] if not mode_val.empty else None

        return self

    def transform(self, X):
        X = self._fix_types(X)
        X = self._validate_schema(X)
        X = X.drop_duplicates().reset_index(drop=True)

        for col, (lower, upper) in self.bounds_.items():
            X[col] = X[col].clip(lower=lower, upper=upper)

        for col, median_val in self.medians_.items():
            X[col] = X[col].fillna(median_val)

        for col, mode_val in self.modes_.items():
            X[col] = X[col].fillna(mode_val)

        return X


## 4. Use It Standalone

In [ ]:
cleaner = DataCleaningTransformer()
cleaned_df = cleaner.fit_transform(raw_df)

print("Raw shape:    ", raw_df.shape)
print("Cleaned shape:", cleaned_df.shape)
cleaned_df.head()


In [ ]:
print("Remaining missing values:\n", cleaned_df.isna().sum())
print("\nRemaining duplicates:", cleaned_df.duplicated().sum())


## 5. Use It Inside a Scikit-learn `Pipeline`

This is the whole point of the bonus challenge: the transformer now plugs straight
into a `Pipeline`, alongside any other scikit-learn step (scalers, encoders, models, etc.).


In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_features = ["age", "income", "purchase_amount", "rating"]

pipeline = Pipeline(steps=[
    ("cleaner", DataCleaningTransformer()),
])

cleaned_via_pipeline = pipeline.fit_transform(raw_df)
print("Cleaned via Pipeline, shape:", cleaned_via_pipeline.shape)
cleaned_via_pipeline.head()


In [ ]:
# Example: extend the pipeline with a scaler on the numeric columns
# (shown separately since our transformer outputs a full DataFrame, not just numeric columns)
scaler = StandardScaler()
scaled_numeric = scaler.fit_transform(cleaned_via_pipeline[numeric_features])

scaled_df = pd.DataFrame(scaled_numeric, columns=numeric_features)
scaled_df.head()


## 6. Quick Visual Check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(x=raw_df["income"], ax=axes[0], color="lightcoral")
axes[0].set_title("Income — Before (Raw)")

sns.boxplot(x=cleaned_via_pipeline["income"], ax=axes[1], color="lightgreen")
axes[1].set_title("Income — After (Pipeline)")

plt.tight_layout()
plt.show()


## 7. Save Output

In [ ]:
bonus_output_path = os.path.join(PROJECT_DIR, "cleaned_customer_data_via_pipeline.csv")
cleaned_via_pipeline.to_csv(bonus_output_path, index=False)
print("Saved:", bonus_output_path)


## 8. Bonus Write-up (also included in `REPORT.md`)

**What I tried:** I wrapped the five-step cleaning pipeline (type-fixing, schema validation,
deduplication, outlier capping, missing-value imputation) into a single
`DataCleaningTransformer` class inheriting from `BaseEstimator` and `TransformerMixin`. The
`fit` step learns medians, modes, and IQR bounds only from the data it's given, and `transform`
applies those learned values — so it behaves correctly inside a `Pipeline` with `fit`/`transform`
on training data and `transform`-only on test data.

**What worked:** The transformer plugs cleanly into `sklearn.pipeline.Pipeline` and produces
identical output to the manual function-based version in the main notebook. Keeping `fit` and
`transform` cleanly separated made it easy to reason about data leakage.

**What didn't work / limitations:** Because the transformer outputs a full DataFrame (including
non-numeric columns like `category` and `signup_date`), it can't sit directly before a plain
`StandardScaler` in the same `Pipeline` without a `ColumnTransformer` — scikit-learn scalers
expect purely numeric input. I handled this by scaling the numeric columns in a separate step
instead of folding everything into one `Pipeline`.

**What I'd do differently with more time:** I would use a `ColumnTransformer` (or
`sklearn.compose.make_column_transformer`) so the entire flow — cleaning, encoding categorical
columns, and scaling numeric columns — runs inside a single `Pipeline.fit`/`.predict` call. I'd
also add unit tests (`pytest`) for each private method (`_fix_types`, `_validate_schema`) to
make the transformer safer to refactor later.
